In [2]:
import pandas as pd
import numpy as np
import torch
import os
import sys
import numpy as np

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)
 
from syn_project.utils_train import *
from syn_project.utils_color_analysis import *
from syn_project.utils_notebook import *

# conditions = [
#     "ablation_cont",
#     "ablation_cont_cy",
#     "ablation_cont_cy_dcy",
#     "ablation_cont_cy_dcy_trans",
#     "ablation_cont_cy_trans",
#     "ablation_cont_dcy",
#     "ablation_cont_dcy_trans",
#     "ablation_cont_trans",
#     "ablation_cy",
#     "ablation_cy_dcy",
#     "ablation_cy_dcy_trans",
#     "ablation_cy_trans",
#     "ablation_dcy",
#     "ablation_dcy_trans",
#     "ablation_trans",
#     "ablation_low_cont",
#     "ablation_low_cy",
#     "ablation_low_dcy",
#     "ablation_low_trans",
# ]

conditions = [
    "action_tr16",

]

start_vision = [True]

In [3]:


checkpoint_epoch  = 0
n_samples_test    = 1000
split             = "test"
dataset           = "biased_00"

rows = []

for condition in conditions:
    print(f"\n=== {condition} ===")
    with total_silence():
        (global_workspace, domain_mods, gw_mod,
         visual_module, original_data,
         latent_domains, modules_name) = get_modules_data_from_exp(
            experiment_name=condition,
            n_samples_test=n_samples_test,
            split=split,
            checkpoint_epoch=checkpoint_epoch,
        )

    for s in start_vision:
        objects = get_objects_from_v_latents(
            latent_domains, gw_mod, global_workspace, modules_name,
            modality_from='attr', modality_through='color',
            modality_main=['attr'], modality_add='color',
            start_v=s,
        )

        original_images_rgb = visual_module.decode_images(original_data['v_latents'])

        cat = []
        if 'attr' in modules_name:
            cat = original_data['attr'][0]
        if 'cat' in modules_name:
            cat = original_data['cat']

        # vision2 = la reconstruction finale qui t'intéresse
        decoded_images = visual_module.decode_images(objects['vision2'])

        # --- qualité de reconstruction ---
        recon = compute_reconstruction_quality(original_images_rgb, decoded_images)

        # --- analyse couleur / LDA ---
        colors_np = np.clip((objects['x2']['color'].detach().cpu().numpy() + 1) / 2, 0, 1) 
        cats      = cat.argmax(dim=1).detach().cpu().numpy()
        metrics, _ = hue_analysis(colors_np, cats,
                                cat_names=CAT_NAMES,
                                value=0.75, saturation_boost=1.8)
        results = logistic_probe(colors_np, cats)

        rows.append({
            "condition"         : condition,
            "start_v"           : s,
            "ssim"              : recon["ssim"],
            "lda_score"         : metrics["lda_score"],
        })

        del decoded_images, objects   # libère la mémoire GPU
        torch.cuda.empty_cache()

# ---- tableau récapitulatif ----

df = pd.DataFrame(rows).set_index("condition")
# df = df.sort_values("lda_score", ascending=False)   # tri par LDA, ajuste si besoin

# affichage Jupyter
display(
    df.style
      .format({"ssim": "{:.3f}",
               "lda_score": "{:.3f}"})
      .background_gradient(subset=["ssim", "lda_score",], cmap="RdYlGn")
)


=== action_tr16 ===


/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


,start_v,ssim,lda_score
condition,,,
action_tr16,True,0.872,0.998


In [4]:
df = pd.DataFrame(rows).set_index(["condition", "start_v"] )
# df = df.sort_values("lda_score", ascending=False)   # tri par LDA, ajuste si besoin

# affichage Jupyter
display(
    df.style
      .format({"ssim": "{:.3f}",
               "lda_score": "{:.3f}"})
      .background_gradient(subset=["ssim", "lda_score",], cmap="RdYlGn")
)

,,ssim,lda_score
condition,start_v,,
action_tr16,True,0.872,0.998


In [5]:
df

,,ssim,lda_score
condition,start_v,,
action_tr16,True,0.872334,0.998
